[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/09.%20The%20Quay%20Crane%20Scheduling%20Problem/P9-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 9. The Quay Crane Scheduling Problem
## Tier 1 — Mixed-Integer Programming (made runnable via exhaustive search)

The Quay Crane Scheduling Problem (QCSP) asks: **how should we assign and sequence quay cranes over bays on a vessel** so that the total completion time (makespan) is as small as possible, while respecting operational constraints.

In the full mathematical treatment, Tier 1 uses a **Mixed-Integer Programming (MIP)** model with binary assignment variables and continuous timing variables. In this notebook we keep the *spirit* of that model but implement a **small, fully runnable example** using exhaustive search instead of an external solver.

### Learning goals
- See how a QCSP-style instance can be written in code (bays, cranes, processing times).
- Understand what the **makespan** is and how it depends on crane assignments and sequences.
- Learn how exhaustive search can play the role of a "pen & paper MIP" for **very small** instances.
- Practice reading tables and visualizations that explain a schedule.

### What this notebook outputs
- An optimal schedule for a small QCSP instance with 4 bays and 2 cranes.
- A table showing, for each crane, which bays it processes and in which order.
- A simple **Gantt-style timeline** that visualizes the schedule.
- A small **what-if experiment** that shows how changing processing times affects makespan.

### Why the instance is small
A full MIP with all QCSP constraints can become very large and needs a dedicated solver. To keep this Tier 1 notebook 100% **runnable with standard scientific Python** and focused on **concepts** rather than solver configuration, we work with a toy instance (4 bays, 2 cranes) and replace the solver with **enumeration of all reasonable assignments and sequences**. The underlying objective is the same as in the MIP: **minimize makespan**.

In [1]:
# Environment check (no installs here)
#
# Best practice for this project: dependencies are preinstalled in the Docker/JupyterHub image.
# If you are running locally, install these once in your virtual environment.

try:
    import itertools
    from dataclasses import dataclass
    from typing import List, Dict, Tuple

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete QCSP instance (4 bays, 2 cranes)

We mirror the small example from the Tier 1 text: a vessel with **4 bays** and **2 quay cranes**.

- Bays: 1, 2, 3, 4
- Processing times (minutes):
  - Bay 1: 45
  - Bay 2: 30
  - Bay 3: 25
  - Bay 4: 35
- Travel time between adjacent bays: 5 minutes

For this small, runnable Tier 1 notebook we make two simplifying assumptions (compared to a full QCSP MIP):
- Each crane processes its assigned bays in **increasing bay index order** (1 → 2 → 3 → 4 along the quay).
- We ignore explicit non-crossing and safety-distance constraints in the code, but keep the **makespan objective** exactly as in the MIP formulation.

These assumptions keep the code simple enough for beginners while still showing how assignment + sequencing → makespan. More advanced Tiers (heuristics, GA, RL, Digital Twin, Multi-Agent, Quantum) build on this intuition.

In [ ]:
# ----------------------------
# Data model for the toy QCSP instance
# ----------------------------
@dataclass(frozen=True)
class Bay:
    id: int          # bay id (1..4)
    position: int    # position index along the quay (1..4)
    process_time: int  # minutes

# Define the 4 bays from the Tier 1 text example
bays: List[Bay] = [
    Bay(id=1, position=1, process_time=45),
    Bay(id=2, position=2, process_time=30),
    Bay(id=3, position=3, process_time=25),
    Bay(id=4, position=4, process_time=35),
]

# Two cranes: we'll refer to them as crane 0 and crane 1.
NUM_CRANES = 2
CRANE_IDS = list(range(NUM_CRANES))

# Travel time between adjacent bay positions (minutes)
TRAVEL_PER_BAY = 5.0

# For convenience, build a few lookup dictionaries.
bay_by_id: Dict[int, Bay] = {b.id: b for b in bays}
bay_ids: List[int] = [b.id for b in bays]
positions = {b.id: b.position for b in bays}
process_times = {b.id: b.process_time for b in bays}

bays, positions, process_times

## Step 1 — Schedules, makespan, and a tiny exact solver

In a QCSP schedule we care about **which crane handles which bays and in what order**.

For this Tier 1 toy example we will:
- Enumerate all **reasonable assignments** of bays to the two cranes.
- For each assignment, assume that each crane processes its bays in **increasing bay position order**.
- Compute for each crane:
  - travel time along the quay,
  - processing time at each bay,
  - total completion time (when that crane finishes).
- The **makespan** is the maximum completion time across cranes.
- We then pick the assignment with the **smallest makespan** as our exact optimum.

This plays the role of the MIP "oracle" on a very small instance: instead of solving a large linear/integer model, we just **test all possibilities**.

In [ ]:
def travel_time(pos_from: int, pos_to: int) -> float:
    """Travel time (minutes) as distance along the quay.

    For this Tier 1 example we assume bays lie on a 1D line and
    travel time is proportional to the absolute position difference.
    """
    return TRAVEL_PER_BAY * abs(pos_from - pos_to)


def crane_timeline_for_sequence(sequence: List[int]) -> Tuple[float, List[Dict]]:
    """Compute completion time and a detailed timeline for ONE crane.

    sequence: list of bay ids in the order this crane will process them.

    Returns: (completion_time, events) where events is a list of dicts
    describing each bay operation: start, finish, travel, process_time.
    """
    if not sequence:
        # This crane stays idle and does no work.
        return 0.0, []

    time = 0.0
    events = []

    # Assume the crane starts positioned at its first bay (no initial travel).
    current_pos = positions[sequence[0]]

    for idx, bay_id in enumerate(sequence):
        bay_pos = positions[bay_id]
        p = process_times[bay_id]

        # Travel from current position to this bay.
        # For the first bay we chose current_pos = bay position, so travel = 0.
        t_travel = travel_time(current_pos, bay_pos)
        time += t_travel
        start_time = time

        # Process the bay.
        time += p
        finish_time = time

        events.append(
            {
                "step": idx + 1,
                "bay_id": bay_id,
                "travel": t_travel,
                "process_time": p,
                "start": start_time,
                "finish": finish_time,
            }
        )

        # Update current position to this bay.
        current_pos = bay_pos

    return time, events


def evaluate_assignment(partition: Dict[int, int]) -> Dict:
    """Evaluate a complete assignment of each bay to a crane.

    partition: mapping bay_id -> crane_id (0 or 1)

    Each crane processes its assigned bays in increasing bay position order.
    We compute a separate timeline for each crane and take the overall makespan.
    """
    # Build sequences per crane, sorted by bay position (left-to-right order)
    crane_sequences: Dict[int, List[int]] = {cid: [] for cid in CRANE_IDS}
    for bay_id in bay_ids:
        cid = partition[bay_id]
        crane_sequences[cid].append(bay_id)

    for cid in CRANE_IDS:
        crane_sequences[cid].sort(key=lambda b: positions[b])

    # Compute timelines for each crane.
    crane_results = {}
    makespan = 0.0
    for cid in CRANE_IDS:
        completion, events = crane_timeline_for_sequence(crane_sequences[cid])
        crane_results[cid] = {
            "sequence": crane_sequences[cid],
            "completion": completion,
            "events": events,
        }
        makespan = max(makespan, completion)

    return {
        "partition": partition,
        "crane_results": crane_results,
        "makespan": makespan,
    }


def exhaustive_solve() -> Dict:
    """Try all 2^4 assignments of the 4 bays to the two cranes.

    For each assignment, we evaluate makespan and return the best one.
    """
    best_result = None
    best_value = float("inf")

    # For each bay we choose crane 0 or 1.
    for choice_tuple in itertools.product(CRANE_IDS, repeat=len(bay_ids)):
        partition = {bay_id: cid for bay_id, cid in zip(bay_ids, choice_tuple)}
        result = evaluate_assignment(partition)
        value = result["makespan"]
        if value < best_value:
            best_value = value
            best_result = result

    assert best_result is not None
    return best_result


best_solution = exhaustive_solve()
best_solution["makespan"]

## Step 2 — Inspect the optimal schedule

A numeric "best makespan" is not very helpful by itself. We now:
- Convert the best solution into a **human-readable table**.
- Check, for each crane: which bays it handles, in what order, and how long it takes.
- Verify that the reported makespan really matches the per-crane completion times.

In [ ]:
# Extract data from the best solution
partition = best_solution["partition"]
crane_results = best_solution["crane_results"]
opt_makespan = best_solution["makespan"]

print(f"Optimal makespan (minutes): {opt_makespan:.1f}")

# Build a flat table of crane operations.
rows = []
for crane_id, info in crane_results.items():
    for event in info["events"]:
        bay_id = event["bay_id"]
        rows.append(
            {
                "crane_id": crane_id,
                "bay_id": bay_id,
                "bay_position": positions[bay_id],
                "process_time": event["process_time"],
                "travel_time": event["travel"],
                "start": event["start"],
                "finish": event["finish"],
            }
        )

schedule_df = pd.DataFrame(rows).sort_values(["crane_id", "start"]).reset_index(drop=True)
schedule_df

In [ ]:
# Sanity check: recompute makespan from the schedule table.
crane_completion_times = (
    schedule_df.groupby("crane_id")["finish"].max().to_dict()
    if not schedule_df.empty
    else {}
)

recomputed_makespan = max(crane_completion_times.values()) if crane_completion_times else 0.0
print("Per-crane completion times (minutes):", crane_completion_times)
print(f"Recomputed makespan (minutes): {recomputed_makespan:.1f}")
print("Matches exhaustive search result:", abs(recomputed_makespan - opt_makespan) < 1e-6)

## Step 3 — Visualizing the schedule (timeline / Gantt view)

A **timeline (Gantt-style)** chart is a natural visualization for crane schedules:
- Each crane is a horizontal line.
- Each bay operation is a colored bar from its start time to finish time.
- The horizontal axis is time (minutes).

This makes it immediately visible which crane is critical for the makespan and where there is idle time.

In [ ]:
def plot_gantt(schedule: pd.DataFrame) -> None:
    """Plot a simple Gantt-style chart for the QCSP schedule."""
    if schedule.empty:
        print("No schedule to plot.")
        return

    fig, ax = plt.subplots(figsize=(8, 3.5))

    colors = {0: "#2E90FA", 1: "#12B76A"}
    yticks = []
    yticklabels = []

    # We place crane 0 at y=0, crane 1 at y=1, etc.
    for crane_id in sorted(schedule["crane_id"].unique()):
        crane_rows = schedule[schedule["crane_id"] == crane_id]
        y = float(crane_id)
        yticks.append(y)
        yticklabels.append(f"Crane {crane_id}")

        for _, row in crane_rows.iterrows():
            start = row["start"]
            finish = row["finish"]
            duration = finish - start
            bay_id = int(row["bay_id"])

            ax.barh(
                y=y,
                width=duration,
                left=start,
                height=0.4,
                color=colors.get(crane_id, "#667085"),
                edgecolor="#101828",
                alpha=0.85,
            )
            ax.text(
                start + duration / 2.0,
                y,
                f"Bay {bay_id}",
                ha="center",
                va="center",
                fontsize=9,
                color="white",
            )

    ax.set_yticks(yticks)
    ax.set_yticklabels(yticklabels)
    ax.set_xlabel("Time (minutes)")
    ax.set_title("Tier 1 QCSP — Gantt-style crane schedule")
    ax.grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()


plot_gantt(schedule_df)

## Step 4 — A small sensitivity / what-if experiment

In practice, processing times are not fixed: they depend on stowage, weather, equipment, and many other factors.

To make the **key idea of Tier 1 (assignment and sequencing)** more visible, we run a small experiment:
- Start from the same 4-bay instance.
- Multiply all processing times by a **scaling factor** (e.g., 0.8, 1.0, 1.2).
- For each factor, recompute the optimal schedule via exhaustive search.
- Compare how the **makespan** changes.

This is not a full stochastic model, but it is enough to show that small changes in processing times can significantly change total completion time.

In [ ]:
def solve_with_scale(scale: float) -> Dict:
    """Re-run the exhaustive solver after scaling all processing times."""
    # Backup original processing times
    original = process_times.copy()

    # Apply scaling
    for bid in process_times.keys():
        process_times[bid] = int(round(original[bid] * scale))

    # Re-run exhaustive search
    result = exhaustive_solve()

    # Restore original processing times and recompute best_solution/global state
    for bid in original.keys():
        process_times[bid] = original[bid]

    return {
        "scale": scale,
        "makespan": result["makespan"],
    }

scales = [0.8, 1.0, 1.2, 1.5]
rows = [solve_with_scale(s) for s in scales]
what_if_df = pd.DataFrame(rows)
what_if_df["makespan"] = what_if_df["makespan"].round(1)
what_if_df

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(what_if_df["scale"], what_if_df["makespan"], marker="o", color="#344054")
plt.xlabel("Processing time scale factor")
plt.ylabel("Optimal makespan (minutes)")
plt.title("Sensitivity of makespan to processing-time scaling (Tier 1)")
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## Step 5 — Interpretation, pitfalls, and link to higher Tiers

### What did we learn from this Tier 1 notebook?
- A QCSP instance can be described in terms of **bays, cranes, processing times, and travel times**.
- The **makespan** is driven by the crane that finishes last.
- Even for 4 bays and 2 cranes, there are multiple possible assignments and sequences.
- Exhaustive search can play the role of a "pen & paper" MIP for very small teaching examples, but it does not scale to real terminals.

### Common pitfalls (and how to debug)
- If the schedule looks strange (e.g., a crane is idle while the other is overloaded):
  - Print `partition` and `crane_results` to see which bays went where.
  - Check the `schedule_df` table carefully.
- If makespan numbers do not match between different parts of the notebook:
  - Re-run the notebook from the top (Kernel → Restart & Run All).
  - Ensure the "what-if" section restored the original processing times.
- If plots are empty:
  - Confirm that `schedule_df` is non-empty and has the expected columns.

### How this Tier connects to higher Tiers
- **Tier 2 (Heuristic)** replaces exhaustive search with a **fast constructive rule** (e.g., an enhanced LPT heuristic) so we can handle more bays and cranes.
- **Tier 3 (Genetic Algorithm)** uses a population of candidate schedules and evolutionary operators to search the solution space more intelligently.
- **Tier 4 (Reinforcement Learning)** turns QCSP into a dynamic decision process and learns dispatching policies from interaction.
- **Tier 5+ (Digital Twin, Multi-Agent, Quantum)** lift the problem into larger system-of-systems and new computing paradigms.

For all higher Tiers, this Tier 1 notebook serves as an **anchor**: it shows, in a simple and fully transparent way, how crane assignments and sequences translate into makespan and how we can compute and visualize a schedule end-to-end.